# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides users in exploring and processing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library, following the Croissant metadata schema for reproducible and standardized data handling.

### Dataset Source
The dataset schema is accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is available (run only if not installed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display summary
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'n/a')}")
print(f"License: {getattr(metadata, 'license', 'n/a')}")

## 2. Data Overview
List available record sets, and inspect their fields and corresponding `@id`s as defined by the Croissant schema.

In [ ]:
# List available record sets and fields (@id reference required)
print("Available record sets (@id | name):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- {rs['@id']} | {rs.get('name', '(no name)')}")

# For demonstration, select the first record set
if record_sets:
    selected_record_set_id = record_sets[0]['@id']
    print(f"\nInspecting fields for record set '{selected_record_set_id}':")
    fields = record_sets[0].get('field', [])
    # field can be a dict (single) or list (multiple)
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  - {field['@id']} | {field.get('name', '(no name)')}")

## 3. Data Extraction
Load the data for all record sets into pandas DataFrames using the `mlcroissant.Dataset.records()` method. All references use the entity `@id` as required.

In [ ]:
# Prepare to extract data for each record set by @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"Warning: No records found for record set '{record_set_id}'.")
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns for the first non-empty record set
for rid, df in dataframes.items():
    if not df.empty:
        print(f"\nFirst few columns for record set '{rid}':")
        print(df.columns.tolist())
        display(df.head())
        main_record_set_id = rid
        break
else:
    raise ValueError("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Let's apply some common data processing steps, such as filtering on a numeric field, normalizing it, and grouping data.

In [ ]:
import numpy as np
# Identify numeric fields from the dataframe columns (using heuristic: look for int, float dtypes)
df_main = dataframes[main_record_set_id]
numeric_cols = df_main.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) == 0:
    print("No numeric columns found for EDA. Showing all columns:")
    print(df_main.columns.tolist())
else:
    numeric_field = numeric_cols[0]
    print(f"Numeric field selected for analysis: {numeric_field}")
    threshold = df_main[numeric_field].median()
    filtered_df = df_main[df_main[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold} (using median as threshold):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by a categorical field, if available
    categorical_cols = df_main.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_cols:
        group_field = categorical_cols[0]
        print(f"\nGrouping by categorical field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions or relationships between fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_cols) > 0:
    # Plot histogram for the main numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df_main[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped data exists, show barplot
    if 'grouped_df' in locals():
        plt.figure(figsize=(8, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
We've demonstrated how to:

- Load and inspect metadata and records from a FAIR² Croissant dataset using `mlcroissant`.
- Programmatically reference all schema entities via their `@id` (`mlcroissant` best practices).
- Load data into pandas, filter on a numeric field, normalize and group, and visualize findings.

**Next steps:** extend to full statistical analysis, model generation, or domain-specific investigation. See [mlcroissant documentation](https://croissant.mlcommons.org/) for more advanced usage!